# Feature Engineering 
This notebook applies Feature Engineering and Data transformation in order to make the data ready for the modell. 
We conduct this step based on the findings of our EDA.
We will apply:
- Frequency based grouping 
- One Hot Encoding
- Time Transformation
- LabelEncoding 

In [43]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.decomposition import PCA

## Load and Merge Datasets
in this section we load both our csv files and merge them into one dataframe

In [44]:
train_id = pd.read_csv("../data/raw/train_identity.csv")
train_trans = pd.read_csv("../data/raw/train_transaction.csv")

df = pd.merge(left=train_trans, right=train_id, how="left",  on="TransactionID")

In [45]:
df.shape

(590540, 434)

## Create Unique Id per User
possible Columns for unique id:
- p_emaildomain = buyer email domain
- addr1 = buyer address (masked)
- card4 = card provider
- card6 = card type
- card1 = id?

now test a few combinations and decide on the one with highest cardinality

In [46]:
df["card4"].unique()

<StringArray>
['discover', 'mastercard', 'visa', 'american express', nan]
Length: 5, dtype: str

In [47]:
df["card6"].unique()

<StringArray>
['credit', 'debit', nan, 'debit or credit', 'charge card']
Length: 5, dtype: str

In [48]:
df["card1"].unique()

array([13926,  2755,  4663, ..., 13166,  8767, 18038], shape=(13553,))

In [49]:
df["uid"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str)
df["uid"].nunique()

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/2789768639.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uid"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str)


37531

In [50]:
df["uid2"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str) + "_" + df["P_emaildomain"] + "_" + df["card4"] + "_" + df["card6"]
df["uid2"].nunique()

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/2338353380.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uid2"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str) + "_" + df["P_emaildomain"] + "_" + df["card4"] + "_" + df["card6"]


73470

In [51]:
df["P_emaildomain"].isna().sum()

np.int64(94456)

In [52]:
df["uid3"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str) + "_" + df["D1"].astype(str)
df["uid3"].nunique()

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/320840475.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uid3"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str) + "_" + df["D1"].astype(str)


235027

In [53]:
df["uid4"] = df["card1"].astype(str) + "_" + df["card6"].astype(str) + df["card4"].astype(str)
df["uid4"].nunique()

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/3606215823.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uid4"] = df["card1"].astype(str) + "_" + df["card6"].astype(str) + df["card4"].astype(str)


13635

we have different options to create an approximated uid. The tradeoff is adding columns with many nan values like addr1 or creating a uid without these but then having less cardinality

## **R_emaildomain -- Relevance Grouping**
we assign numeric values 1-5 for the top 5 email provider used in our dataset. As we do have many nullvalues (80%) and have other uncommon domains, we group those together under the numeric value 6

In [54]:
top5_email_categories = df["R_emaildomain"].value_counts().head(5)

In [55]:
email_mapper = {}
for i, domain in enumerate(top5_email_categories.index.tolist()):
    email_mapper[domain] = i+1

In [56]:
df["R_emaildomain"] = df["R_emaildomain"].map(email_mapper).fillna(6).astype(int)

## **One-Hot-Encoding**
in this section we take multiple string columns with only a few (<10) different values and apply one hot encoding

### ProductCD Column

In [57]:
df = pd.get_dummies(data=df, prefix="ProductCD", columns=["ProductCD"])

### card4 Column

In [58]:
df = pd.get_dummies(data=df, columns=["card4"], prefix="card4")

### card6 Column

In [59]:
df = pd.get_dummies(data=df, prefix="card6", columns=["card6"])

### DeviceType column

In [60]:
df = pd.get_dummies(data=df, prefix="DeviceType", columns=["DeviceType"])

## Time Column 
as we found out in our EDA, the "TransactionDT" corresponds to the unit second.
Therefore we can create a weekday (0-6) and a hour (0-23) column.
This allows us as humans to properly interpret the data, while also enabling the Modell to find patterns more easily

In [61]:
df["DT_day"] = df["TransactionDT"] // 86400
df["DT_hour"] = (df["TransactionDT"] // 3600) % 24
df["DT_weekday"] = df["DT_day"] % 7

df = df.drop(["DT_day", "TransactionDT"], axis=1)

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/3280605987.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_day"] = df["TransactionDT"] // 86400
/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/3280605987.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_hour"] = (df["TransactionDT"] // 3600) % 24
/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_72888/3280605987.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.ins

## Ordinal Encoder
we want to make sure that NaN values are being natively handled by XGBoost. Therefore we want to remain those nan values during LabelEncoding. To ensure NaN values are being kept, we mask only the non NaN values and apply the transformation exclusively for those. We do so by creating a Series, mapping via boolean if a given Value is a string or nan(string = true, Nan = false). Then we create a new, NaN filled column (encoded). All index of mask in encoded now being transformed by our label encoder to become numeric values.

In [73]:
cat_cols = df.select_dtypes(include="str").columns.tolist()

enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=np.nan
)

df[cat_cols] = enc.fit_transform(df[cat_cols])

## **PCA for v Columns**
Because of high correlation of the 300+ V columns, we decided to merge these information together by applying PCA

In [72]:
pca = PCA(n_components=4)

v_cols = [col for col in df.columns if col.startswith("V")]

pca_result = pca.fit_transform(df[v_cols].fillna(0))

print(f"Percentage of variance explained: {pca.explained_variance_ratio_.sum():.2%}")

Percentage of variance explained: 99.65%


As already seen in our EDA, the V columns correlate heaviliy with each other. This allows us to merge the information together to 4 new axis which provide us with 99.65 of variance from the original 300+ cols. This helps the model to remove noise.